In [28]:
import pandas as pd

In [20]:
# Define file paths for ads and articles datasets
ads_dataset_paths = [
    "../datasets/ads_dataset_24-05-2024.csv",
    "../datasets/ads_dataset_30-05-2024.csv",
    "../datasets/ads_dataset_04-06-2024.csv",
    "../datasets/ads_dataset_07-06-2024.csv",
]

articles_dataset_paths = [
    "../datasets/articles_dataset_24-05-2024.csv",
    "../datasets/articles_dataset_30-05-2024.csv",
    "../datasets/articles_dataset_04-06-2024.csv",
    "../datasets/articles_dataset_07-06-2024.csv",
]

# Combine all file paths into a single list for easier processing
all_dataset_paths = ads_dataset_paths + articles_dataset_paths

# Read all datasets into DataFrames and store them in a dictionary
dataframes = {}
for i, file_path in enumerate(all_dataset_paths, start=1):
    # Read CSV file into a DataFrame
    dataframes[f'df_{i}'] = pd.read_csv(file_path)

In [21]:
# Concatenate ads DataFrames into a single DataFrame
ads_df = pd.concat([dataframes[f'df_{i}'] for i in range(1, 5)], ignore_index=True)

In [5]:
# Data Quality Assessment

# Check for missing values in the ads DataFrame
missing_values = ads_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Get a summary of total missing values in the DataFrame
total_missing_values = ads_df.isnull().sum().sum()
print(f"Total number of missing values: {total_missing_values}")

# Check for duplicate entries in the ads DataFrame
duplicates = ads_df.duplicated()

# Get the number of duplicate rows
num_duplicates = duplicates.sum()
print(f"Number of duplicate rows: {num_duplicates}")

Missing values in each column:
Folder_Number       0
Image_Link          0
Ad_Name            77
Product           123
Company           189
Ad_Description     25
Ad_Placement        3
dtype: int64
Total number of missing values: 417
Number of duplicate rows: 27


In [6]:
# Remove duplicate rows from the ads DataFrame and reset the index
ads_df = ads_df.drop_duplicates().reset_index(drop=True)

# Remove rows where 'Ad Name', 'Product', and 'Company' are all null
ads_df = ads_df[~ads_df[['Ad_Name', 'Product', 'Company']].isnull().all(axis=1)]

print(ads_df.isnull().sum())

Folder_Number       0
Image_Link          0
Ad_Name            33
Product            72
Company           143
Ad_Description      9
Ad_Placement        0
dtype: int64


In [7]:
# Define a list of unwanted ad names and companies to filter out
unwanted_ad_names = [
    'Google Ads',
    'Ads by Google',
    'Advertisement from Google',
    'Ad using Google',
    'Google Ads Placeholder',
    'Google Ad',
    'Anuncio servido por Google'
]

# Remove rows with unwanted ad names
ads_df = ads_df[~ads_df['Ad_Name'].isin(unwanted_ad_names)]

# Remove rows where the company is 'Google'
ads_df = ads_df[ads_df['Company'] != 'Google']

In [22]:
# Dictionary containing translations of relevant keywords related to elections and political campaigns 
# in multiple languages. These keywords will be used to filter the ad dataset.
keywords_translations = {
    "English": ["election", "political campaign", "voting", "vote", "poll", "candidate", "campaign"],
    "Spanish": ["elección", "campaña política", "votación", "votar", "encuesta", "candidato", "campaña"],
    "German": ["Wahl", "Wahlkampf", "Abstimmung", "wählen", "Umfrage", "Kandidat", "Kampagne"],
    "French": ["élection", "campagne politique", "vote", "voter", "sondage", "candidat", "campagne"],
    "Italian": ["elezione", "campagna politica", "voto", "votare", "sondaggio", "candidato", "campagna"],
    "Dutch": ["verkiezing", "politieke campagne", "stemming", "stemmen", "peiling", "kandidaat", "campagne"],
    "Portuguese": ["eleição", "campanha política", "votação", "votar", "pesquisa", "candidato", "campanha"],
    "Polish": ["wybory", "kampania polityczna", "głosowanie", "głosować", "sondaż", "kandydat", "kampania"],
    "Swedish": ["val", "politisk kampanj", "omröstning", "rösta", "opinionsundersökning", "kandidat", "kampanj"],
    "Danish": ["valg", "politisk kampagne", "afstemning", "stemme", "meningsmåling", "kandidat", "kampagne"],
    "Finnish": ["vaalit", "poliittinen kampanja", "äänestys", "äänestää", "mielipidetutkimus", "ehdokas", "kampanja"],
    "Greek": ["εκλογή", "πολιτική εκστρατεία", "ψηφοφορία", "ψηφίζω", "δημοσκόπηση", "υποψήφιος", "εκστρατεία"],
    "Hungarian": ["választás", "politikai kampány", "szavazás", "szavaz", "közvéleménykutatás", "jelölt", "kampány"]
}

# Initialize an empty list to store all the translated keywords from the dictionary
search_strings = []

# Iterate over each language (key) and its corresponding list of keywords (value)
for key, value in keywords_translations.items():
    # Append each keyword to the search_strings list
    for word in value:
        search_strings.append(word)

# Create a search pattern that combines all keywords into a single string separated by the '|' character
# This allows searching for any of these keywords in a case-insensitive manner
pattern = '|'.join(search_strings)

# Filter the ads DataFrame by checking if any of the fields 'Ad_Name', 'Product', 'Company', or 'Ad_Description'
# contain one or more of the keywords (ignoring case and treating missing values as False)
filtered_df = ads_df[
    ads_df['Ad_Name'].str.contains(pattern, case=False, na=False) |
    ads_df['Product'].str.contains(pattern, case=False, na=False) |
    ads_df['Company'].str.contains(pattern, case=False, na=False) |
    ads_df['Ad_Description'].str.contains(pattern, case=False, na=False)
]

In [27]:
filtered_df.to_csv("../datasets/political_ads.csv", index =False)

### Once the file is created the wrong rows which are not related to politics/elections are filtered manually and removed from the file